<a href="https://colab.research.google.com/github/rprakash-rec/Gen-AI_Lab-AD23633/blob/main/Exp_5B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf sentence-transformers faiss-cpu numpy transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.6/335.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.8 MB/s eta 0:00:00


In [2]:
from pypdf import PdfReader

In [3]:
from sentence_transformers import SentenceTransformer

In [4]:
import faiss
import numpy as np # FAISS requires float32 numpy arrays

In [5]:
from transformers import pipeline
from typing import List # safer than list[str] for Python < 3.10
from textwrap import wrap as word_wrap

In [8]:
def load_pdf(path: str) -> str:
  reader = PdfReader(path)
  full_text = ""
  for page in reader.pages:
    full_text += page.extract_text() or ""
  return full_text

def chunk_text(text: str,
               chunk_size: int = 500,
               overlap: int = 50) -> List[str]:
  words = text.split() # split on whitespace
  chunks = []
  start = 0
  while start < len(words):
    end = start + chunk_size
    chunk = " ".join(words[start:end])
    chunks.append(chunk)
    start += chunk_size - overlap # slide window forward
  return chunks

In [12]:
def build_embeddings(chunks: List[str],
 model_name: str = "all-MiniLM-L6-v2"):
 print(f" Loading embedding model: {model_name} ...")
 embedder = SentenceTransformer(model_name)
 print(f" Embedding {len(chunks)} chunks ...")
 embeddings = embedder.encode(chunks,show_progress_bar=True)
 return embedder, embeddings.astype("float32")

In [13]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
 dimension = embeddings.shape[1]
 index = faiss.IndexFlatL2(dimension)
 index.add(embeddings)
 print(f" FAISS index built: {index.ntotal} vectors stored.")
 return index

In [16]:
def retrieve(query: str,
             embedder,
             index: faiss.Index,
             chunks: List[str],
             top_k: int = 3) -> List[str]:
  query_vec = embedder.encode([query]).astype("float32")
  distances, indices = index.search(query_vec, top_k)
  retrieved_chunks = [chunks[idx] for idx in indices[0] if idx != -1]
  return retrieved_chunks

In [18]:
def generate_answer(query: str,
 context_chunks: List[str],
 generator,
 max_tokens: int = 300) -> str:
  context = "\n\n---\n\n".join(context_chunks)
  prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say
\"I don't know based on the provided documents.\"
Context:
{context}
Question: {query}
Answer:"""
  result = generator(prompt, max_new_tokens=max_tokens)
  return result[0]["generated_text"].strip()

In [19]:
def build_rag_pipeline(pdf_paths: List[str]):
  all_chunks = []
  for pdf_path in pdf_paths:
    print(f"\n[+] Loading: {pdf_path}")
    text = load_pdf(pdf_path)
    chunks = chunk_text(text, chunk_size=500, overlap=50)
    print(f" -> {len(chunks)} chunks extracted")
    all_chunks.extend(chunks)
  print(f"\n[+] Total chunks: {len(all_chunks)}")
  print("\n[+] Building embeddings ...")
  embedder, embeddings = build_embeddings(all_chunks)
  print("\n[+] Building FAISS index ...")
  index = build_faiss_index(embeddings)
  return embedder, index, all_chunks

def ask(query: str, embedder, index, chunks, generator, top_k: int = 3):
  print(f"\n{'='*60}")
  print(f"QUESTION: {query}")
  print(f"{'='*60}")
  print("\n[Retrieve] Searching FAISS index ...")
  context = retrieve(query, embedder, index, chunks, top_k=top_k)
  print(f" -> {len(context)} chunks retrieved")
  print("\n[Generate] Calling flan-t5-base ...")
  answer = generate_answer(query, context, generator)
  print("\nANSWER:\n")
  for line in word_wrap(answer, width=70):
    print(" ", line)
  return answer

In [24]:
PDF_FILES = [
 "5b_pdf.pdf", # change to your filename
 "5b_pdf2.pdf", # change to your filename
]
embedder, index, chunks = build_rag_pipeline(PDF_FILES)
print("\n[+] Loading local LLM: flan-t5-base ...")
generator = pipeline(
 "text2text-generation",
 model="google/flan-t5-base"
)
questions = [
 "What is the core idea behind the attention mechanism?",
 "How does BERT differ from traditional language models?",
 "What datasets were used for pre-training?",
]
for q in questions:
  ask(q, embedder, index, chunks, generator)


[+] Loading: 5b_pdf.pdf
 -> 14 chunks extracted

[+] Loading: 5b_pdf2.pdf
 -> 23 chunks extracted

[+] Total chunks: 37

[+] Building embeddings ...
 Loading embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

 Embedding 37 chunks ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[+] Building FAISS index ...
 FAISS index built: 37 vectors stored.

[+] Loading local LLM: flan-t5-base ...


config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"